---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-31: LangChain <b>Prompts</b> Component</h1>

# Learning agenda of this notebook

> **The idea of this lecture:** in Lec-30 we wrote every prompt by hand, as a fixed string. Real apps send the **same kind of prompt** again and again with different values: a different review, a different topic, a different chat history. A **prompt template** is that prompt written once, with named blanks that are filled in at run time. Templates are the second building block of an LLM app, and the chains (Lec-33) and RAG apps (Lec-38) are built on them.

1. **Where Prompts Fit**
2. **What is a Prompt Template?**
    - Setup: the model
3. **`PromptTemplate`: One Text with Blanks**
    - Example 1: An f-string vs a `PromptTemplate`
    - Example 2: `format()` vs `invoke()`, and sending the prompt to the model
    - Example 3: One template for many inputs, and `partial()`
    - Example 4: Saving a template to a file and loading it back
4. **`ChatPromptTemplate`: Role Messages with Blanks**
    - Example 5: A system message and a human message with variables
    - Example 6: `MessagesPlaceholder`: a slot for the chat history
5. **Where Prompt Templates Go Next**
6. **Summary**

> **Note:** all examples use the free `openai/gpt-oss-120b` on Groq (Lec-30, Example 3).

# <span style='background :lightgreen' >1. Where Prompts Fit</span>

In the app of Lec-30, the **prompt template** sits between the retriever and the chat model: it puts the question and the retrieved text into one well-structured prompt.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

> **Documentation:** https://reference.langchain.com/python/langchain-core/prompts (the classes of this lecture live in `langchain_core.prompts`).

# <span style='background :lightgreen' >2. What is a Prompt Template?</span>

- A **prompt template** is a prompt with **named placeholders** in curly braces, such as `{review}` or `{topic}`. It is written once and **filled in** with different values at run time.
- LangChain has two kinds: **`PromptTemplate`** for one piece of text, and **`ChatPromptTemplate`** for a list of messages with roles (system, human, AI).
- Filling a template with `invoke()` gives a **PromptValue**, which every chat model accepts.

<div style="text-align:center;">
    <img src="../images/langchain-new-11.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## Setup: the model
The same Groq model as in Lec-30 (Example 3), created once and used in every example.

In [1]:
import os                                                   # read GROQ_API_KEY
from dotenv import load_dotenv                              # load the API keys from ../keys/.env
from langchain_openai import ChatOpenAI                     # works with Groq's OpenAI-compatible API (Lec-30, Example 3)

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ (the file wins)
print("GROQ_API_KEY", "set" if os.getenv("GROQ_API_KEY") else "MISSING")   # never print any part of a key

GROQ = {"base_url": "https://api.groq.com/openai/v1", "api_key": os.getenv("GROQ_API_KEY"), "max_retries": 5}
GROQ_MODEL = ChatOpenAI(model="openai/gpt-oss-120b", **GROQ)   # free; no max_tokens, so its reasoning never cuts the answer

GROQ_API_KEY set


# <span style='background :lightgreen' >3. <code>PromptTemplate</code>: One Text with Blanks</span>

## <span style='color:#7e22ce'>Example 1: An f-string vs a <code>PromptTemplate</code></span>
### Step 1: An f-string
A Python f-string is filled **the moment it is written**: the value must already exist, and the result is a finished string. To use other values we must write the f-string again (or wrap it in a function).

In [2]:
user_name = "Arif"                                          # the value must exist BEFORE the f-string is written
prompt = f"Greet {user_name} in five languages, one of which should be Urdu. One line per language."
print(prompt)                                               # already a finished string

print(GROQ_MODEL.invoke(prompt).text)                       # send it to the model

Greet Arif in five languages, one of which should be Urdu. One line per language.
Hello Arif!  
¡Hola Arif!  
Bonjour Arif!  
سلام ارِف!  
Hallo Arif!


### Step 2: A `PromptTemplate`
- `PromptTemplate.from_template(...)` keeps the text and finds the placeholders by itself: `input_variables`.
- The **same** template is filled again for each new value.
- A **missing** value raises a clear `KeyError` that names it, instead of quietly sending a broken prompt.
- The template uses `{name}` placeholders only: it fills them in and runs **no** Python code. (To keep a literal brace in the text, write it twice: `{{` and `}}`.)

In [3]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders

greet_template = PromptTemplate.from_template(
    "Greet {user_name} in five languages, one of which should be Urdu. One line per language.")
print("Placeholders:", greet_template.input_variables)     # found automatically from the {braces}

for name in ["Arif", "Ayesha"]:                             # the same template, two values
    print("\n" + GROQ_MODEL.invoke(greet_template.invoke({"user_name": name})).text)

try:
    greet_template.invoke({})                               # forgot the value
except KeyError as error:
    print("\nKeyError:", str(error)[:80], "…")              # names the missing variable

Placeholders: ['user_name']

Hello, Arif!  
¡Hola, Arif!  
Bonjour, Arif!  
ہیلو، عارف!  
مرحبا، عارف!

Hello Ayesha!  
آئیشا کو سلام!  
¡Hola Ayesha!  
Bonjour Ayesha !  
مرحبًا عائشة!  

KeyError: "Input to PromptTemplate is missing variables {'user_name'}.  Expected: ['user_n …


## <span style='color:#7e22ce'>Example 2: <code>format()</code> vs <code>invoke()</code>, and sending the prompt to the model</span>
| Method | Takes | Returns | Use it to |
|---|---|---|---|
| `template.format(topic=..., audience=...)` | keyword arguments | a plain `str` | print or log the prompt |
| `template.invoke({"topic": ..., "audience": ...})` | a dict | a `StringPromptValue` | send it to a model; `.to_string()` and `.to_messages()` show what is inside |

`invoke()` is the method that every LangChain component has: templates, models, and (in Lec-33) chains. That is what lets us connect them.

In [4]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders

joke_template = PromptTemplate.from_template("Tell me a short joke about {topic} for {audience}.")

as_text = joke_template.format(topic="parrots", audience="elders")          # keyword arguments → str
print(type(as_text).__name__, "→", as_text)

as_value = joke_template.invoke({"topic": "parrots", "audience": "elders"})  # a dict → PromptValue
print(type(as_value).__name__, "→", as_value.to_string())
print("As messages:", as_value.to_messages())               # what the chat model will receive: one HumanMessage

print("\n" + GROQ_MODEL.invoke(as_value).text)              # a chat model accepts the PromptValue directly

str → Tell me a short joke about parrots for elders.
StringPromptValue → Tell me a short joke about parrots for elders.
As messages: [HumanMessage(content='Tell me a short joke about parrots for elders.', additional_kwargs={}, response_metadata={})]

Why did the elderly parrot join the knitting club?  

Because it heard they were great at “tweet‑and‑purl”! 🪡🐦


## <span style='color:#7e22ce'>Example 3: One template for many inputs, and <code>partial()</code></span>
### Step 1: A sentiment classifier for many reviews
One template, five reviews: we fill it five times and send all five prompts with `batch()` (Lec-30, Example 8).

In [5]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders

sentiment_template = PromptTemplate.from_template(
    "Classify the sentiment of this review as Positive, Negative or Mixed. Reply with one word.\nReview: {review}")

reviews = ["The delivery was late but the product quality is excellent.",
           "Absolutely love it, works perfectly!",
           "Stopped working after two days. Waste of money.",
           "Good camera, terrible battery.",
           "Fast shipping and great customer service."]

prompts = [sentiment_template.invoke({"review": review}) for review in reviews]   # five filled prompts
for review, reply in zip(reviews, GROQ_MODEL.batch(prompts)):   # all five sent at the same time
    print(f"{reply.text:9} ← {review}")

Mixed     ← The delivery was late but the product quality is excellent.
Positive  ← Absolutely love it, works perfectly!
Negative  ← Stopped working after two days. Waste of money.
Mixed     ← Good camera, terrible battery.
Positive  ← Fast shipping and great customer service.


### Step 2: `partial()`: fill some blanks now, the rest later
`partial()` returns a new template with some values already filled in. Here we make an Urdu version and an Arabic version of one greeting template.

In [6]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders

greeting = PromptTemplate.from_template("Write a one-line greeting for {name} in {language}, with an English translation.")
urdu_greeting = greeting.partial(language="Urdu")           # language is fixed; only {name} is left
arabic_greeting = greeting.partial(language="Arabic")

print("Still needed:", urdu_greeting.input_variables)       # ['name']
for template in [urdu_greeting, arabic_greeting]:
    print(GROQ_MODEL.invoke(template.invoke({"name": "Arif"})).text, "\n")

Still needed: ['name']
**اردو:** ارِف، آپ کو دل سے خوش آمدید!  
**English:** Arif, a warm welcome to you! 

مرحبًا يا عارف! — Hello, Arif! 



## <span style='color:#7e22ce'>Example 4: Saving a template to a file and loading it back</span>
- A good prompt is worth keeping: we save it once and load it in other programs, or share it with the team.
- `dumps()` turns any LangChain object into JSON text, and `loads()` turns the JSON back into the object. (The older `template.save()` and `load_prompt()` are deprecated since langchain-core 1.2.21 and will be removed in 2.0.)
- **Loading builds Python objects from a file**, so we tell `loads()` exactly which classes it may build: `allowed_objects=[PromptTemplate]`. A file from someone else can then never create anything else.
- `loads()` is still marked *beta*, so we hide its beta warning.

In [7]:
import warnings                                             # hide a warning we expect
from pathlib import Path                                    # the file path
from langchain_core._api import LangChainBetaWarning        # the warning class LangChain uses for beta features
from langchain_core.load import dumps, loads                # LangChain object ↔ JSON text
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders

warnings.filterwarnings("ignore", category=LangChainBetaWarning)   # loads() is in beta

greet_template = PromptTemplate.from_template(
    "Greet {user_name} in five languages, one of which should be Urdu. One line per language.")

path = Path("../data/greet_template.json")
path.write_text(dumps(greet_template, pretty=True))         # 1. save: object → JSON text → file
print(path.read_text(), "\n")                               # what is inside the file

loaded = loads(path.read_text(), allowed_objects=[PromptTemplate])   # 2. load: file → object (PromptTemplate only)
print("Loaded:", type(loaded).__name__, loaded.input_variables, "\n")
print(GROQ_MODEL.invoke(loaded.invoke({"user_name": "Arif"})).text)  # 3. use it as before

{
  "lc": 1,
  "type": "constructor",
  "id": [
    "langchain",
    "prompts",
    "prompt",
    "PromptTemplate"
  ],
  "kwargs": {
    "input_variables": [
      "user_name"
    ],
    "template": "Greet {user_name} in five languages, one of which should be Urdu. One line per language.",
    "template_format": "f-string"
  },
  "name": "PromptTemplate"
} 

Loaded: PromptTemplate ['user_name'] 

Hello Arif (English)  
¡Hola Arif! (Spanish)  
Bonjour Arif (French)  
नमस्ते अरीफ़ (Hindi)  
سلام عارف (Urdu)


# <span style='background :lightgreen' >4. <code>ChatPromptTemplate</code>: Role Messages with Blanks</span>

- A chat model receives a **list of messages with roles** (Lec-30, Example 6). A `ChatPromptTemplate` is that list, with blanks.
- Each message is a `(role, text)` pair. The roles are `"system"`, `"human"` (or `"user"`) and `"ai"` (or `"assistant"`).
- A **`MessagesPlaceholder`** is a slot for a whole list of messages, such as the chat history.

<div style="text-align:center;">
    <img src="../images/langchain-new-12.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 5: A system message and a human message with variables</span>
The role of the assistant is a variable too, so one template gives us a tutor and a comedian.

In [8]:
from langchain_core.prompts import ChatPromptTemplate       # a list of (role, text) templates

chat_template = ChatPromptTemplate([
    ("system", "You are a {role}. Answer in at most two sentences."),   # the instructions, with a blank
    ("human", "{question}"),                                            # the user's message, with a blank
])
print("Placeholders:", chat_template.input_variables)

prompt_value = chat_template.invoke({"role": "patient Python tutor", "question": "What is a list comprehension?"})
for message in prompt_value.to_messages():                  # a ChatPromptValue: real message objects
    print(f"{type(message).__name__:13} {message.content}")

for role in ["patient Python tutor", "stand-up comedian"]:  # one template, two personas
    reply = GROQ_MODEL.invoke(chat_template.invoke({"role": role, "question": "What is a list comprehension?"}))
    print(f"\n[{role}] {reply.text}")

Placeholders: ['question', 'role']
SystemMessage You are a patient Python tutor. Answer in at most two sentences.
HumanMessage  What is a list comprehension?

[patient Python tutor] A list comprehension is a compact syntax for creating a new list by evaluating an expression over each item in an iterable, optionally filtering with an `if` clause; e.g., `[x*2 for x in nums if x>0]`. It combines looping and optional conditionals into a single readable line.

[stand-up comedian] A list comprehension is Python’s way of saying, “Hey, I want a whole list, but I don’t want to write a novel of loops”—basically a one‑liner that builds a list on the fly. Think of it as the fast‑food version of a for‑loop: quick, tasty, and you don’t have to clean up the kitchen afterward.


## <span style='color:#7e22ce'>Example 6: <code>MessagesPlaceholder</code>: a slot for the chat history</span>
- In Lec-30 (Example 7) we kept the history in a list and sent all of it. With a template, the **fixed** parts (the instructions) stay in the template, and the history goes into the `MessagesPlaceholder("history")` slot.
- After each turn we add the user's message and the model's answer to `history`.
- `("placeholder", "{history}")` is a short way to write the same slot. In Lec-32, LangChain will keep the history for us.

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder   # templates, and a slot for messages
from langchain.messages import HumanMessage                 # a message written by the user

chatbot_template = ChatPromptTemplate([
    ("system", "You are a friendly assistant. Reply in one or two sentences."),
    MessagesPlaceholder("history"),                         # the chat so far goes here
    ("human", "{question}"),                                # the new question
])

history = []                                                # starts empty
for question in ["Hi, I'm Arif and I teach Data Science.", "I also like cricket.", "What's my name, and what do I teach?"]:
    prompt_value = chatbot_template.invoke({"history": history, "question": question})   # fill both blanks
    reply = GROQ_MODEL.invoke(prompt_value)
    print(f"You:   {question}\nModel: {reply.text}   [{len(prompt_value.to_messages())} messages sent]\n")
    history += [HumanMessage(question), reply]              # remember this turn: the question and the answer

You:   Hi, I'm Arif and I teach Data Science.
Model: Nice to meet you, Arif! Teaching data science must be rewarding—what topics do you enjoy sharing most with your students?   [2 messages sent]

You:   I also like cricket.
Model: That's great! Do you have a favorite cricket team or a memorable match you love talking about?   [4 messages sent]

You:   What's my name, and what do I teach?
Model: Your name is Arif, and you teach Data Science.   [6 messages sent]



# <span style='background :lightgreen' >5. Where Prompt Templates Go Next</span>

| Where | What the template does there | Lecture |
|---|---|---|
| **Chains** | `prompt | model | parser`: the template is the first step of a chain | Lec-33 |
| **Memory** | LangChain fills the `MessagesPlaceholder` with the history for us | Lec-32 |
| **RAG** | A `ChatPromptTemplate` with `{context}` (the retrieved chunks) and `{question}` | Lec-38 |
| **MCP servers** | A server can offer ready-made prompt templates; clients list and fill them | Lec-46, 47 |
| **Agents** | `create_agent(system_prompt=...)`, and prompts that change per call with `@dynamic_prompt` middleware | Lec-51, 52 |

# <span style='background :lightgreen' >6. Summary</span>

| Without templates (Lec-30) | With templates |
|---|---|
| An f-string, filled once, when it is written | `PromptTemplate.from_template("... {name} ...")`, filled again and again |
| A wrong or missing value goes unnoticed | A missing value raises a `KeyError` that names it |
| The same prompt copied into every program | `dumps()` to a file, `loads(..., allowed_objects=[...])` back |
| A list of messages built by hand | `ChatPromptTemplate([("system", ...), ("human", ...)])` |
| The history appended by hand into the list | `MessagesPlaceholder("history")` |

### Key takeaways
- A **prompt template** is a prompt written once, with named `{placeholders}` that are filled in at run time.
- **`PromptTemplate`** is for one piece of text; **`ChatPromptTemplate`** is for a list of role messages.
- `format()` returns a `str`; `invoke()` returns a **PromptValue** that any chat model accepts.
- `partial()` fills some placeholders now and the rest later.
- Save templates with `dumps()` and load them with `loads()`, allowing only the classes you expect.
- **`MessagesPlaceholder`** is a slot for a list of messages, such as the chat history.
- **Next lecture (Lec-32):** the **Memory** component: keeping the chat history for us.

### Check your understanding
1. What is the difference between an f-string and a `PromptTemplate`? When is each one filled in?
2. What does `input_variables` contain, and where does it come from?
3. What happens if a value is missing? Why is that better than an f-string with a wrong variable?
4. What do `format()` and `invoke()` return? Which one would you pass to `GROQ_MODEL.invoke()`?
5. In Example 3, how many times was the template filled, and how many model calls were made at the same time?
6. What does `partial(language="Urdu")` return? Which placeholders are still left?
7. Why do we pass `allowed_objects=[PromptTemplate]` to `loads()`?
8. What are the three roles of a `ChatPromptTemplate`, and how are they written?
9. In Example 6, how many messages were sent on the third turn, and why?